In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler
import time
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, matthews_corrcoef
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('unsw_nb15.csv',low_memory=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 49 columns):
 #   Column            Dtype  
---  ------            -----  
 0   srcip             object 
 1   sport             object 
 2   dstip             object 
 3   dsport            object 
 4   proto             object 
 5   state             object 
 6   dur               float64
 7   sbytes            int64  
 8   dbytes            int64  
 9   sttl              int64  
 10  dttl              int64  
 11  sloss             int64  
 12  dloss             int64  
 13  service           object 
 14  Sload             float64
 15  Dload             float64
 16  Spkts             int64  
 17  Dpkts             int64  
 18  swin              int64  
 19  dwin              int64  
 20  stcpb             int64  
 21  dtcpb             int64  
 22  smeansz           int64  
 23  dmeansz           int64  
 24  trans_depth       int64  
 25  res_bdy_len       int64  
 26  Sjit          

In [3]:
df.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0


In [4]:
df.isnull().sum()

srcip                     0
sport                     0
dstip                     0
dsport                    0
proto                     0
state                     0
dur                       0
sbytes                    0
dbytes                    0
sttl                      0
dttl                      0
sloss                     0
dloss                     0
service                   0
Sload                     0
Dload                     0
Spkts                     0
Dpkts                     0
swin                      0
dwin                      0
stcpb                     0
dtcpb                     0
smeansz                   0
dmeansz                   0
trans_depth               0
res_bdy_len               0
Sjit                      0
Djit                      0
Stime                     0
Ltime                     0
Sintpkt                   0
Dintpkt                   0
tcprtt                    0
synack                    0
ackdat                    0
is_sm_ips_ports     

In [5]:
df['attack_cat'].unique()

array([nan, 'Exploits', 'Reconnaissance', 'DoS', 'Generic', 'Shellcode',
       ' Fuzzers', 'Worms', 'Backdoors', 'Analysis', ' Reconnaissance ',
       'Backdoor', ' Fuzzers ', ' Shellcode '], dtype=object)

In [6]:
# Dictionary for correction
correction_dict = {
    np.nan: 'Normal',
    'Exploits': 'Exploits',
    'Reconnaissance': 'Reconnaissance',
    'DoS': 'DoS',
    'Generic': 'Generic',
    'Shellcode': 'Shellcode',
    ' Fuzzers': 'Fuzzers',
    'Worms': 'Worms',
    'Backdoors': 'Backdoor',
    'Analysis': 'Analysis',
    ' Reconnaissance ': 'Reconnaissance',
    'Backdoor': 'Backdoor',
    ' Fuzzers ': 'Fuzzers',
    ' Shellcode ': 'Shellcode'
}

# Apply the correction
df['attack_cat'] = df['attack_cat'].replace(correction_dict)

# Now use value_counts()
count_series = df['attack_cat'].value_counts()

print(count_series)

attack_cat
Normal            2218764
Generic            215481
Exploits            44525
Fuzzers             24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoor             2329
Shellcode            1511
Worms                 174
Name: count, dtype: int64


In [7]:
df[df['is_ftp_login'].notna()]['attack_cat'].value_counts()

attack_cat
Normal            1086068
Generic              7526
Exploits             7137
Fuzzers              5182
Reconnaissance       1759
DoS                  1189
Backdoor              534
Analysis              526
Shellcode             223
Worms                  24
Name: count, dtype: int64

In [8]:
df.loc[df['ct_ftp_cmd'] == ' ', 'ct_ftp_cmd'] = 7
df['ct_ftp_cmd'].value_counts()

ct_ftp_cmd
7    1429879
0    1066498
1      40077
2       1264
4        960
3        729
6        332
5        290
8         18
Name: count, dtype: int64

In [9]:
df['ct_ftp_cmd'] = pd.to_numeric(df['ct_ftp_cmd'])

In [10]:
def hex_to_int(value):
    try:
        # Convert hex string to integer
        return int(value, 16)
    except ValueError:
        # If the value is not a valid hex, return the original value
        return value

# Apply the hex_to_int function to the 'sport' and 'dsport' columns
df['sport'] = df['sport'].apply(hex_to_int)
df['dsport'] = df['dsport'].apply(hex_to_int)

# Now convert the columns to numeric dtype
df['sport'] = pd.to_numeric(df['sport'], errors='coerce')
df['dsport'] = pd.to_numeric(df['dsport'], errors='coerce')

print(df[['sport', 'dsport']].head())

      sport  dsport
0    5008.0    83.0
1  210529.0  4132.0
2    5220.0    83.0
3   13715.0    83.0
4  300644.0    83.0


In [11]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
df.describe()

,sport,dsport,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,Sload,Dload,Spkts,Dpkts,swin,dwin,stcpb,dtcpb,smeansz,dmeansz,trans_depth,res_bdy_len,Sjit,Djit,Stime,Ltime,Sintpkt,Dintpkt,tcprtt,synack,ackdat,is_sm_ips_ports,ct_state_ttl,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,Label
count,2.540045e+06,2.540040e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,1.191902e+06,1.110168e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06
mean,1.881021e+05,6.819673e+04,6.587916e-01,4.339600e+03,3.642759e+04,6.278197e+01,3.076681e+01,5.163921e+00,1.632944e+01,3.695645e+07,2.450861e+06,3.328884e+01,4.272664e+01,1.500887e+02,1.497459e+02,1.261701e+09,1.261766e+09,1.242536e+02,2.766719e+02,8.325318e-02,4.242118e+03,1.589037e+03,7.300755e+02,1.423261e+09,1.423261e+09,1.933225e+02,7.882476e+01,6.180475e-03,3.287595e-03,2.892880e-03,1.651544e-03,2.611546e-01,2.345856e-01,3.969940e-02,3.961096e+00,9.206988e+00,8.988958e+00,6.439103e+00,6.900986e+00,4.642139e+00,3.592729e+00,6.845886e+00,1.264870e-01
std,1.318857e+05,4.922397e+05,1.392493e+01,5.640599e+04,1.610960e+05,7.462277e+01,4.285089e+01,2.251707e+01,5.659474e+01,1.186043e+08,4.224863e+06,7.628388e+01,1.215020e+02,1.254824e+02,1.255438e+02,1.422027e+09,1.422139e+09,1.519162e+02,3.356166e+02,3.500045e-01,4.750053e+04,1.691036e+04,3.438558e+03,1.134449e+06,1.134448e+06,2.779163e+03,1.433191e+03,4.615863e-02,2.593570e-02,2.394668e-02,4.060563e-02,6.830974e-01,7.940924e-01,1.996589e-01,3.453679e+00,1.083676e+01,1.082249e+01,8.162034e+00,8.205062e+00,8.477579e+00,6.174445e+00,1.125828e+01,3.323975e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.421927e+09,1.421927e+09,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,7.018200e+04,8.300000e+01,1.037000e-03,2.000000e+02,1.780000e+02,3.100000e+01,2.900000e+01,0.000000e+00,0.000000e+00,1.353963e+05,1.191594e+04,2.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.000000e+01,6.900000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.421952e+09,1.421952e+09,9.000000e-03,6.000000e-03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
50%,2.023750e+05,1.280000e+02,1.586100e-02,1.470000e+03,1.820000e+03,3.100000e+01,2.900000e+01,3.000000e+00,4.000000e+00,5.893038e+05,5.893179e+05,1.200000e+01,1.200000e+01,2.550000e+02,2.550000e+02,6.397250e+08,6.384172e+08,7.300000e+01,8.900000e+01,0.000000e+00,0.000000e+00,1.912490e+01,2.653561e+00,1.424227e+09,1.424227e+09,4.682620e-01,4.147550e-01,6.130000e-04,4.830000e-04,1.220000e-04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.000000e+00,5.000000e+00,5.000000e+00,3.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,0.000000e+00
75%,2.918970e+05,8.432300e+04,2.145545e-01,3.182000e+03,1.489400e+04,3.100000e+01,2.900000e+01,7.000000e+00,1.400000e+01,2.039923e+06,2.925974e+06,4.400000e+01,4.200000e+01,2.550000e+02,2.550000e+02,2.467160e+09,2.469411e+09,1.320000e+02,5.650000e+02

In [12]:
df.isnull().sum()

srcip                     0
sport                     2
dstip                     0
dsport                    7
proto                     0
state                     0
dur                       0
sbytes                    0
dbytes                    0
sttl                      0
dttl                      0
sloss                     0
dloss                     0
service                   0
Sload                     0
Dload                     0
Spkts                     0
Dpkts                     0
swin                      0
dwin                      0
stcpb                     0
dtcpb                     0
smeansz                   0
dmeansz                   0
trans_depth               0
res_bdy_len               0
Sjit                      0
Djit                      0
Stime                     0
Ltime                     0
Sintpkt                   0
Dintpkt                   0
tcprtt                    0
synack                    0
ackdat                    0
is_sm_ips_ports     

In [13]:
df.drop(['ct_flw_http_mthd','is_ftp_login'],axis=1,inplace=True)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 47 columns):
 #   Column            Dtype  
---  ------            -----  
 0   srcip             object 
 1   sport             float64
 2   dstip             object 
 3   dsport            float64
 4   proto             object 
 5   state             object 
 6   dur               float64
 7   sbytes            int64  
 8   dbytes            int64  
 9   sttl              int64  
 10  dttl              int64  
 11  sloss             int64  
 12  dloss             int64  
 13  service           object 
 14  Sload             float64
 15  Dload             float64
 16  Spkts             int64  
 17  Dpkts             int64  
 18  swin              int64  
 19  dwin              int64  
 20  stcpb             int64  
 21  dtcpb             int64  
 22  smeansz           int64  
 23  dmeansz           int64  
 24  trans_depth       int64  
 25  res_bdy_len       int64  
 26  Sjit          

In [15]:
import hashlib

# Function to hash an IP address
def hash_ip(ip):
    return int(hashlib.sha256(ip.encode()).hexdigest(), 16) % 10**8

# Apply the hashing function to the 'srcip' column
df['srcip_hashed'] = df['srcip'].apply(hash_ip)

# Apply the hashing function to the 'dstip' column
df['dstip_hashed'] = df['dstip'].apply(hash_ip)

# Display the first few rows to check the hashed IP addresses
print(df[['srcip', 'dstip', 'srcip_hashed', 'dstip_hashed']].head())

        srcip          dstip  srcip_hashed  dstip_hashed
0  59.166.0.0  149.171.126.6      72552287      94553001
1  59.166.0.0  149.171.126.9      72552287      29903251
2  59.166.0.6  149.171.126.7      92005674        204114
3  59.166.0.5  149.171.126.5      29218943      86602698
4  59.166.0.3  149.171.126.0      39972581      26014570


In [16]:
# Define port bins
bins = [0, 1023, 49151, 65535]
labels = ['Well-known', 'Registered', 'Dynamic/private']

# Bin source and destination ports
df['sport_bin'] = pd.cut(df['sport'], bins=bins, labels=labels, include_lowest=True)
df['dsport_bin'] = pd.cut(df['dsport'], bins=bins, labels=labels, include_lowest=True)

# One-hot encode the port bins
df = pd.get_dummies(df, columns=['sport_bin', 'dsport_bin'], prefix=['sport', 'dsport'])

print(df.head())

        srcip     sport          dstip  dsport proto state       dur  sbytes  \
0  59.166.0.0    5008.0  149.171.126.6    83.0   udp   CON  0.001055     132   
1  59.166.0.0  210529.0  149.171.126.9  4132.0   udp   CON  0.036133     528   
2  59.166.0.6    5220.0  149.171.126.7    83.0   udp   CON  0.001119     146   
3  59.166.0.5   13715.0  149.171.126.5    83.0   udp   CON  0.001209     132   
4  59.166.0.3  300644.0  149.171.126.0    83.0   udp   CON  0.001169     146   

   dbytes  sttl  dttl  sloss  dloss service         Sload         Dload  \
0     164    31    29      0      0     dns  500473.93750  621800.93750   
1     304    31    29      0      0       -   87676.08594   50480.17188   
2     178    31    29      0      0     dns  521894.53130  636282.37500   
3     164    31    29      0      0     dns  436724.56250  542597.18750   
4     178    31    29      0      0     dns  499572.25000  609067.56250   

   Spkts  Dpkts  swin  dwin  stcpb  dtcpb  smeansz  dmeansz  trans_d

In [17]:
# Get the last 6 columns of the DataFrame
boolean_columns = df.columns[-6:]

# Convert boolean columns to numeric
df[boolean_columns] = df[boolean_columns].astype(int)

df.head()


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,service,Sload,Dload,Spkts,Dpkts,swin,dwin,stcpb,dtcpb,smeansz,dmeansz,trans_depth,res_bdy_len,Sjit,Djit,Stime,Ltime,Sintpkt,Dintpkt,tcprtt,synack,ackdat,is_sm_ips_ports,ct_state_ttl,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label,srcip_hashed,dstip_hashed,sport_Well-known,sport_Registered,sport_Dynamic/private,dsport_Well-known,dsport_Registered,dsport_Dynamic/private
0,59.166.0.0,5008.0,149.171.126.6,83.0,udp,CON,0.001055,132,164,31,29,0,0,dns,500473.93750,621800.93750,2,2,0,0,0,0,66,82,0,0,0.00000,0.000000,1421927414,1421927414,0.017,0.013000,0.0,0.0,0.0,0,0,0,3,7,1,3,1,1,1,Normal,0,72552287,94553001,0,1,0,1,0,0
1,59.166.0.0,210529.0,149.171.126.9,4132.0,udp,CON,0.036133,528,304,31,29,0,0,-,87676.08594,50480.17188,4,4,0,0,0,0,132,76,0,0,9.89101,10.682733,1421927414,1421927414,7.005,7.564333,0.0,0.0,0.0,0,0,0,2,4,2,3,1,1,2,Normal,0,72552287,29903251,0,0,0,0,1,0
2,59.166.0.6,5220.0,149.171.126.7,83.0,udp,CON,0.001119,146,178,31,29,0,0,dns,521894.53130,636282.37500,2,2,0,0,0,0,73,89,0,0,0.00000,0.000000,1421927414,1421927414,0.017,0.013000,0.0,0.0,0.0,0,0,0,12,8,1,2,2,1,1,Normal,0,92005674,204114,0,1,0,1,0,0
3,59.166.0.5,13715.0,149.171.126.5,83.0,udp,CON,0.001209,132,164,31,29,0,0,dns,436724.56250,542597.18750,2,2,0,0,0,0,66,82,0,0,0.00000,0.000000,1421927414,1421927414,0.043,0.014000,0.0,0.0,0.0,0,0,0,6,9,1,1,1,1,1,Normal,0,29218943,86602698,0,1,0,1,0,0
4,59.166.0.3,300644.0,149.171.126.0,83.0,udp,CON,0.001169,146,178,31,29,0,0,dns,499572.25000,609067.56250,2,2,0,0,0,0,73,89,0,0,0.00000,0.000000,1421927414,1421927414,0.005,0.003000,0.0,0.0,0.0,0,0,0,7,9,1,1,1,1,1,Normal,0,39972581,26014570,0,0,0,1,0,0


In [18]:
# Get number of unique values for 'proto' and 'state'
num_unique_proto = df['proto'].nunique()
num_unique_state = df['state'].nunique()

# Get value counts for 'proto' and 'state'
value_counts_proto = df['proto'].value_counts()
value_counts_state = df['state'].value_counts()

# Print results
print(f"Number of unique values in 'proto': {num_unique_proto}")
print(f"Value counts for 'proto':\n{value_counts_proto}\n")
print(f"Number of unique values in 'state': {num_unique_state}")
print(f"Value counts for 'state':\n{value_counts_state}")


Number of unique values in 'proto': 135
Value counts for 'proto':
proto
tcp            1495074
udp             990435
unas             16202
arp              10064
ospf              7798
sctp              1525
icmp               524
any                411
gre                324
rsvp               274
ipv6               272
swipe              262
sun-nd             262
pim                262
mobile             262
sep                260
micp               137
aes-sp3-d          137
encap              137
eigrp              137
ipip               137
ax.25              137
mtp                137
larp               137
sprite-rpc         137
pri-enc            137
etherip            137
vmtp               137
tcf                137
dgp                137
nsfnet-igp         137
ttp                137
vines              137
pnni               137
secure-vmtp        137
iso-ip             137
wb-expak           137
wb-mon             137
br-sat-mon         137
pvp                137
wsn     

In [19]:
def process_column(df, column, threshold):
    # Get the value counts of the column
    value_counts = df[column].value_counts()

    # Identify the values that meet or exceed the threshold
    frequent_values = value_counts[value_counts >= threshold].index

    # Replace infrequent values with 'others'
    df[column] = df[column].where(df[column].isin(frequent_values), other='other')

    return df

threshold = 500

# Apply the process_column function for each categorical column

df = process_column(df, 'state', threshold)
df = process_column(df, 'proto', threshold)

# Now use OneHotEncoder for one-hot encoding with drop='first' and sparse_output=False
encoder = OneHotEncoder(sparse_output=False)

# Fit and transform the three processed categorical columns
encoded_columns = encoder.fit_transform(df[['state', 'proto']])

# Get the names of the new one-hot encoded columns
encoded_column_names = encoder.get_feature_names_out(['state', 'proto'])

# Create a DataFrame from the one-hot encoded data
encoded_df = pd.DataFrame(encoded_columns, columns=encoded_column_names, index=df.index)

# Concatenate the encoded columns back to the original DataFrame (dropping the original categorical columns)
df = pd.concat([df.drop(['service', 'state', 'proto'], axis=1), encoded_df], axis=1)


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 66 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   srcip                   object 
 1   sport                   float64
 2   dstip                   object 
 3   dsport                  float64
 4   dur                     float64
 5   sbytes                  int64  
 6   dbytes                  int64  
 7   sttl                    int64  
 8   dttl                    int64  
 9   sloss                   int64  
 10  dloss                   int64  
 11  Sload                   float64
 12  Dload                   float64
 13  Spkts                   int64  
 14  Dpkts                   int64  
 15  swin                    int64  
 16  dwin                    int64  
 17  stcpb                   int64  
 18  dtcpb                   int64  
 19  smeansz                 int64  
 20  dmeansz                 int64  
 21  trans_depth             int64  

In [21]:
df.shape

(2540047, 66)

In [22]:
# Drop the first 4 columns by column names
df = df.drop(columns=df.columns[:4])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 62 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   dur                     float64
 1   sbytes                  int64  
 2   dbytes                  int64  
 3   sttl                    int64  
 4   dttl                    int64  
 5   sloss                   int64  
 6   dloss                   int64  
 7   Sload                   float64
 8   Dload                   float64
 9   Spkts                   int64  
 10  Dpkts                   int64  
 11  swin                    int64  
 12  dwin                    int64  
 13  stcpb                   int64  
 14  dtcpb                   int64  
 15  smeansz                 int64  
 16  dmeansz                 int64  
 17  trans_depth             int64  
 18  res_bdy_len             int64  
 19  Sjit                    float64
 20  Djit                    float64
 21  Stime                   int64  

In [23]:
df= df.drop('Label',axis=1)

In [24]:
# Select only the numeric columns
numeric_columns = df.select_dtypes(include=['float64', 'int64','int32']).columns

# Initialize MinMaxScaler with the desired feature range (0, 1)
scaler = MinMaxScaler()

# Fit and transform only the numeric columns
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 61 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   dur                     float64
 1   sbytes                  float64
 2   dbytes                  float64
 3   sttl                    float64
 4   dttl                    float64
 5   sloss                   float64
 6   dloss                   float64
 7   Sload                   float64
 8   Dload                   float64
 9   Spkts                   float64
 10  Dpkts                   float64
 11  swin                    float64
 12  dwin                    float64
 13  stcpb                   float64
 14  dtcpb                   float64
 15  smeansz                 float64
 16  dmeansz                 float64
 17  trans_depth             float64
 18  res_bdy_len             float64
 19  Sjit                    float64
 20  Djit                    float64
 21  Stime                   float64

In [25]:
# Initialize the LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the 'attack_cat' column
df['attack_cat'] = label_encoder.fit_transform(df['attack_cat'])

# Show class mapping (mapping between original categories and assigned integers)
class_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

# Print the class mapping
print("Class Mapping:", class_mapping)

Class Mapping: {'Analysis': 0, 'Backdoor': 1, 'DoS': 2, 'Exploits': 3, 'Fuzzers': 4, 'Generic': 5, 'Normal': 6, 'Reconnaissance': 7, 'Shellcode': 8, 'Worms': 9}


In [26]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop('attack_cat', axis=1)  # Features
y = df['attack_cat']  # Target

# Step 1: Split the data into 70% training and 30% temporary set (for validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Step 2: Split the temporary set into 50% validation and 50% test (15% each of the original dataset)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Print the shapes of the resulting splits
print("Shape of X_train:", X_train.shape)  # 70% of the data
print("Shape of X_val:", X_val.shape)      # 15% of the data
print("Shape of X_test:", X_test.shape)    # 15% of the data
print("Shape of y_train:", y_train.shape)
print("Shape of y_val:", y_val.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (1778032, 60)
Shape of X_val: (381007, 60)
Shape of X_test: (381008, 60)
Shape of y_train: (1778032,)
Shape of y_val: (381007,)
Shape of y_test: (381008,)


In [27]:
class_mapping = {0: 'Analysis',
                1: 'Backdoor',
                2: 'DoS',
                3: 'Exploits',
                4: 'Fuzzers',
                5: 'Generic',
                6: 'Normal',
                7: 'Reconnaissance',
                8: 'Shellcode',
                9: 'Worms'}

# Train and validate XGBoost
def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test):
    model = XGBClassifier(tree_method='hist', device='cuda', eval_metric="mlogloss",
                          random_state=42, num_classes=5, objective="multi:softmax", max_depth=15,
                          n_estimators=200, learning_rate=0.2)

    # Training the model without early stopping
    start_time = time.time()
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose = False)
    training_time = time.time() - start_time

    print("\nEvaluating on Validation Set:")
    evaluate_xgb(model, X_val, y_val)

    print("\nEvaluating on Test Set:")
    evaluate_xgb(model, X_test, y_test)

    print('\nTraining Time: {:.2f} seconds'.format(training_time))

# Function to evaluate the model (for both validation and test sets)
def evaluate_xgb(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred, digits=4, target_names=[class_mapping[i] for i in range(len(class_mapping))]))

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_eval, y_pred)
    print(cm)

    print("\nROC-AUC Score:", roc_auc_score(pd.get_dummies(y_eval), model.predict_proba(X_eval), multi_class="ovr"))
    print("\nMatthews Correlation Coefficient:", matthews_corrcoef(y_eval, y_pred))


print("Training XGBoost Classifier...")
train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test)

Training XGBoost Classifier...

Evaluating on Validation Set:

Classification Report:
                precision    recall  f1-score   support

      Analysis     0.7143    0.1247    0.2123       401
      Backdoor     0.7021    0.0946    0.1667       349
           DoS     0.3634    0.3074    0.3330      2453
      Exploits     0.6498    0.8167    0.7238      6679
       Fuzzers     0.7532    0.6866    0.7184      3637
       Generic     0.9971    0.9880    0.9925     32322
        Normal     0.9974    0.9981    0.9977    332815
Reconnaissance     0.9141    0.7707    0.8363      2098
     Shellcode     0.8031    0.8987    0.8482       227
         Worms     0.3571    0.1923    0.2500        26

      accuracy                         0.9835    381007
     macro avg     0.7252    0.5878    0.6079    381007
  weighted avg     0.9837    0.9835    0.9829    381007


Confusion Matrix:
[[    50      1     90    174     28      2     56      0      0      0]
 [     2     33     90    190     2

In [28]:
y_train.value_counts()

attack_cat
6    1553134
5     150837
3      31167
4      16972
2      11447
7       9791
0       1874
1       1630
8       1058
9        122
Name: count, dtype: int64

In [29]:
from imblearn.over_sampling import SMOTE

# --------------------------
# Step 1: Duplicate minority classes in the training set
# --------------------------
# Combine the features and target into one DataFrame for easier manipulation
df_train = X_train.copy()
df_train['attack_type'] = y_train

# Separate the training data by class
df_train_0 = df_train[df_train['attack_type'] == 0]
df_train_1 = df_train[df_train['attack_type'] == 1]
df_train_2 = df_train[df_train['attack_type'] == 2]
df_train_3 = df_train[df_train['attack_type'] == 3]
df_train_4 = df_train[df_train['attack_type'] == 4]
df_train_5 = df_train[df_train['attack_type'] == 5]
df_train_6 = df_train[df_train['attack_type'] == 6]
df_train_7 = df_train[df_train['attack_type'] == 7]
df_train_8 = df_train[df_train['attack_type'] == 8]
df_train_9 = df_train[df_train['attack_type'] == 9]

# Duplicate:
# - For class 9: repeat each sample 10 times
# - For class 8: repeat each sample 5 times
# - For class 1: repeat each sample 4 times
# - For class 0: repeat each sample 4 times
# - For class 7 and 2: repeat each sample 2 times
df_train_9_dup = pd.concat([df_train_9] * 10, ignore_index=True)
df_train_8_dup = pd.concat([df_train_8] * 5, ignore_index=True)
df_train_1_dup = pd.concat([df_train_1] * 4, ignore_index=True)
df_train_0_dup = pd.concat([df_train_0] * 4, ignore_index=True)
df_train_7_dup = pd.concat([df_train_7] * 2, ignore_index=True)
df_train_2_dup = pd.concat([df_train_2] * 2, ignore_index=True)

# Combine the duplicated minority classes with the rest of the training data
df_train_aug = pd.concat([df_train_3, df_train_4, df_train_5, df_train_6,
                          df_train_0_dup, df_train_1_dup, df_train_2_dup,
                          df_train_7_dup, df_train_8_dup, df_train_9_dup], ignore_index=True)

print("Training set shape after duplication:", df_train_aug.shape)
print("Class distribution after duplication:")
print(df_train_aug['attack_type'].value_counts())

# --------------------------
# Step 2: Apply SMOTE oversampling on the training set (after duplication)
# --------------------------
# Prepare features and target
X_train_aug = df_train_aug.drop(columns=['attack_type'])
y_train_aug = df_train_aug['attack_type']

# Define the SMOTE sampling strategy:
# Increase classes 9, 8, 1, 0 to 20,000 samples, and classes 4, 2, 7, 3 to 40,000 samples
smote_strategy = {
    0: 20000,
    1: 20000,
    2: 40000,
    3: 40000,
    4: 40000,
    7: 40000,
    8: 20000,
    9: 20000
}

smote = SMOTE(sampling_strategy=smote_strategy, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_aug, y_train_aug)

print("Training set shape after SMOTE:", X_train_smote.shape)
print("Class distribution after SMOTE:")
unique, counts = np.unique(y_train_smote, return_counts=True)
print(dict(zip(unique, counts)))

print("\nEvaluating model trained on SMOTE oversampled training data:")
train_xgboost(X_train_smote, y_train_smote, X_val, y_val, X_test, y_test)

Training set shape after duplication: (1815112, 61)
Class distribution after duplication:
attack_type
6    1553134
5     150837
3      31167
2      22894
7      19582
4      16972
0       7496
1       6520
8       5290
9       1220
Name: count, dtype: int64
Training set shape after SMOTE: (1943971, 60)
Class distribution after SMOTE:
{0: 20000, 1: 20000, 2: 40000, 3: 40000, 4: 40000, 5: 150837, 6: 1553134, 7: 40000, 8: 20000, 9: 20000}

Evaluating model trained on SMOTE oversampled training data:

Evaluating on Validation Set:

Classification Report:
                precision    recall  f1-score   support

      Analysis     0.1616    0.1596    0.1606       401
      Backdoor     0.1111    0.1691    0.1341       349
           DoS     0.3365    0.7020    0.4550      2453
      Exploits     0.8313    0.5835    0.6857      6679
       Fuzzers     0.7788    0.7055    0.7403      3637
       Generic     0.9980    0.9877    0.9928     32322
        Normal     0.9977    0.9979    0.9978    3

In [33]:
# Combine X_train_smote and y_train_smote into a single DataFrame for easier manipulation
train_smote_df = pd.concat([X_train_smote, y_train_smote], axis=1)

# Assuming the target column for class labels in y_train_smote is named 'class'
# Adjust the name 'class' to match your actual column name in y_train_smote if it's different
class_column = y_train_smote.name  # Automatically detects the name of the class column

# Class distribution (your provided support for each class)
class_support = {0: 20000, 1: 20000, 2: 40000, 3: 40000, 4: 40000, 5: 150837, 6: 1553134, 7: 40000, 8: 20000, 9: 20000}

# Target support for classes 5 and 6
target_support = 60000

# Undersampling classes 5 and 6
class_5_indices = train_smote_df[train_smote_df[class_column] == 5].index
class_6_indices = train_smote_df[train_smote_df[class_column] == 6].index

undersampled_class_5_indices = np.random.choice(class_5_indices, size=target_support, replace=False)
undersampled_class_6_indices = np.random.choice(class_6_indices, size=target_support, replace=False)

# Concatenate the undersampled indices with the other classes
all_indices = []
for class_label in class_support.keys():
    if class_label == 5:
        all_indices.extend(undersampled_class_5_indices)
    elif class_label == 6:
        all_indices.extend(undersampled_class_6_indices)
    else:
        all_indices.extend(train_smote_df[train_smote_df[class_column] == class_label].index)

# Create a new DataFrame with the selected indices
undersampled_df = train_smote_df.loc[all_indices]

# Split back into X_train_smote and y_train_smote
X_train_undersampled = undersampled_df.drop(columns=[class_column])
y_train_undersampled = undersampled_df[class_column]

# Check the new class distribution
print(y_train_undersampled.value_counts())
print("\nEvaluating model trained on undersampled training data:")
train_xgboost(X_train_undersampled, y_train_undersampled, X_val, y_val, X_test, y_test)

attack_type
5    60000
6    60000
2    40000
3    40000
4    40000
7    40000
0    20000
1    20000
8    20000
9    20000
Name: count, dtype: int64

Evaluating model trained on undersampled training data:

Evaluating on Validation Set:

Classification Report:
                precision    recall  f1-score   support

      Analysis     0.1281    0.2743    0.1746       401
      Backdoor     0.1063    0.1777    0.1330       349
           DoS     0.3337    0.7028    0.4526      2453
      Exploits     0.7739    0.5898    0.6694      6679
       Fuzzers     0.4556    0.8661    0.5971      3637
       Generic     0.9989    0.9829    0.9909     32322
        Normal     0.9999    0.9868    0.9933    332815
Reconnaissance     0.8830    0.7841    0.8306      2098
     Shellcode     0.7589    0.9427    0.8409       227
         Worms     0.5217    0.4615    0.4898        26

      accuracy                         0.9738    381007
     macro avg     0.5960    0.6769    0.6172    381007
  weighted

In [36]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
import json
import datetime
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import numpy as np

##############################################
# Model Saver Class (Local PC)
##############################################

class LocalModelSaver:
    def __init__(self, base_path='./models'):
        self.base_path = Path(base_path)
        self.base_path.mkdir(parents=True, exist_ok=True)

    def save_checkpoint(self, generator, discriminator,
                        opt_g, opt_d, epoch, metrics,
                        model_name='gan_model'):
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        save_dir = self.base_path / model_name / timestamp
        save_dir.mkdir(parents=True, exist_ok=True)

        checkpoint = {
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_g_state_dict': opt_g.state_dict(),
            'optimizer_d_state_dict': opt_d.state_dict(),
            'metrics': metrics
        }

        torch.save(checkpoint, save_dir / 'checkpoint.pt')

        config = {
            'generator_config': {
                'latent_dim': generator.latent_dim,
                'label_dim': generator.label_dim
            },
            'epoch': epoch,
            'timestamp': timestamp
        }

        with open(save_dir / 'config.json', 'w') as f:
            json.dump(config, f, indent=4)

        print(f"Model saved successfully at: {save_dir}")
        return str(save_dir)

    def load_checkpoint(self, model_path, generator, discriminator,
                        opt_g=None, opt_d=None):
        checkpoint = torch.load(Path(model_path) / 'checkpoint.pt')
        generator.load_state_dict(checkpoint['generator_state_dict'])
        discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
        if opt_g is not None:
            opt_g.load_state_dict(checkpoint['optimizer_g_state_dict'])
        if opt_d is not None:
            opt_d.load_state_dict(checkpoint['optimizer_d_state_dict'])
        return checkpoint['epoch'], checkpoint['metrics']

    def save_quick(self, generator, discriminator, model_name='gan_model'):
        save_dir = self.base_path / model_name / 'quick_save'
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save(generator.state_dict(), save_dir / 'generator.pt')
        torch.save(discriminator.state_dict(), save_dir / 'discriminator.pt')
        print(f"Quick save completed at: {save_dir}")
        return str(save_dir)
##############################################
# Model Definitions
##############################################

class Generator(nn.Module):
    def __init__(self, latent_dim, label_dim, output_dim):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim
        self.label_dim = label_dim
        self.initial = nn.Linear(latent_dim + label_dim, 128)
        self.hidden1 = nn.Linear(128, 256)
        self.hidden2 = nn.Linear(256, 512)
        self.output = nn.Linear(512, output_dim)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, z, labels):
        if len(z.shape) == 1:
            z = z.unsqueeze(0)
        if len(labels.shape) == 1:
            labels = labels.unsqueeze(0)
        x = torch.cat((z, labels), dim=1)
        x = self.leaky(self.initial(x))
        x = self.leaky(self.hidden1(x))
        x = self.leaky(self.hidden2(x))
        return torch.sigmoid(self.output(x))

class Discriminator(nn.Module):
    def __init__(self, input_dim, label_dim):
        super(Discriminator, self).__init__()
        self.input_dim = input_dim
        self.label_dim = label_dim
        self.initial = nn.Linear(input_dim + label_dim, 512)
        self.hidden1 = nn.Linear(512, 256)
        self.hidden2 = nn.Linear(256, 128)
        self.output = nn.Linear(128, 1)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, x, labels):
        if len(x.shape) == 1:
            x = x.unsqueeze(0)
        if len(labels.shape) == 1:
            labels = labels.unsqueeze(0)
        x = torch.cat((x, labels), dim=1)
        x = self.leaky(self.initial(x))
        x = self.leaky(self.hidden1(x))
        x = self.leaky(self.hidden2(x))
        return self.output(x)

##############################################
# Helper Classes
##############################################

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False

    def __call__(self, current_loss):
        if self.best_loss is None:
            self.best_loss = current_loss
            return False
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

##############################################
# Training Functions
##############################################

def train_gan(generator, discriminator, train_loader, val_loader, device, model_saver,
              num_epochs=200, save_interval=200, patience=5, min_delta=0.001):

    generator = generator.to(device)
    discriminator = discriminator.to(device)

    optimizer_G = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    scaler = GradScaler()

    early_stopping = EarlyStopping(patience=patience, min_delta=min_delta)

    history = {
        'g_loss': [],
        'd_loss': [],
        'val_g_loss': [],
        'val_d_loss': []
    }

    best_g_loss = float('inf')

    for epoch in range(num_epochs):
        generator.train()
        discriminator.train()

        total_g_loss = 0
        total_d_loss = 0
        batches_processed = 0

        for real_data, real_labels in train_loader:
            current_batch_size = real_data.size(0)
            real_data = real_data.to(device)
            real_labels = real_labels.to(device)

            with autocast():
                z = torch.randn(current_batch_size, generator.initial.in_features - real_labels.size(1), device=device)
                fake_data = generator(z, real_labels)
                d_real = discriminator(real_data, real_labels)
                d_fake = discriminator(fake_data.detach(), real_labels)

                d_loss_real = F.binary_cross_entropy_with_logits(
                    d_real, torch.ones_like(d_real, device=device) * 0.9)
                d_loss_fake = F.binary_cross_entropy_with_logits(
                    d_fake, torch.zeros_like(d_fake, device=device) + 0.1)
                d_loss = (d_loss_real + d_loss_fake) / 2

            optimizer_D.zero_grad(set_to_none=True)
            scaler.scale(d_loss).backward()
            scaler.step(optimizer_D)
            scaler.update()

            total_d_loss += d_loss.item()

            for _ in range(3):
                optimizer_G.zero_grad(set_to_none=True)
                with autocast():
                    z = torch.randn(current_batch_size, generator.initial.in_features - real_labels.size(1), device=device)
                    fake_data = generator(z, real_labels)
                    fake_output = discriminator(fake_data, real_labels)
                    g_loss = F.binary_cross_entropy_with_logits(
                        fake_output, torch.ones_like(fake_output, device=device) * 0.9)
                scaler.scale(g_loss).backward()
                scaler.step(optimizer_G)
                scaler.update()
                total_g_loss += g_loss.item()

            batches_processed += 1

        avg_g_loss = total_g_loss / (batches_processed * 3)
        avg_d_loss = total_d_loss / batches_processed

        history['g_loss'].append(avg_g_loss)
        history['d_loss'].append(avg_d_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] G_loss: {avg_g_loss:.4f} D_loss: {avg_d_loss:.4f}")

        if (epoch + 1) % save_interval == 0:
            val_g_loss, val_d_loss = evaluate_model(generator, discriminator, val_loader, device)

            metrics = {
                'g_loss': avg_g_loss,
                'd_loss': avg_d_loss,
                'val_g_loss': val_g_loss,
                'val_d_loss': val_d_loss
            }

            save_path = model_saver.save_checkpoint(
                generator, discriminator,
                optimizer_G, optimizer_D,
                epoch, metrics
            )

            if val_g_loss < best_g_loss:
                best_g_loss = val_g_loss

            if early_stopping(val_g_loss):
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                break

    final_path = model_saver.save_quick(generator, discriminator)
    return history


##############################################
# Data Preparation Functions
##############################################

def prepare_data(X, y, batch_size=1000):
    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(y, pd.Series):
        y = y.values

    encoder = OneHotEncoder(sparse_output=False)
    y_one_hot = encoder.fit_transform(y.reshape(-1, 1))

    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y_one_hot, dtype=torch.float32)
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    return loader, encoder, y_one_hot.shape[1]

def generate_synthetic_data(generator, num_samples, label_dim, device, X_train, y_train):
    generator.eval()
    synthetic_data = []
    synthetic_labels = []

    with torch.no_grad():
        for class_idx in range(label_dim):
            labels = torch.zeros(num_samples, label_dim, device=device)
            labels[:, class_idx] = 1

            z = torch.randn(num_samples, generator.initial.in_features - label_dim, device=device)
            fake_data = generator(z, labels).cpu().numpy()

            synthetic_data.append(fake_data)
            synthetic_labels.append(np.full(num_samples, class_idx))

    X_synthetic = pd.DataFrame(np.vstack(synthetic_data), columns=X_train.columns)
    y_synthetic = pd.DataFrame(np.hstack(synthetic_labels), columns=y_train.columns)
    
    return X_synthetic, y_synthetic

##############################################
# Main Function
##############################################

def main():
    torch.manual_seed(42)
    np.random.seed(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Initialize the model saver
    model_saver = LocalModelSaver()

    batch_size = 1000
    latent_dim = 200
    num_epochs = 200

    print(f"Training data shapes - X: {X_train_undersampled.shape}, y: {y_train_undersampled.shape}")

    # Prepare data
    train_loader, encoder, num_classes = prepare_data(X_train_undersampled, y_train_undersampled, batch_size)
    val_loader, _, _ = prepare_data(X_val, y_val, batch_size)

    print(f"Number of features: {X_train_undersampled.shape[1]}")
    print(f"Number of classes: {num_classes}")
    print(f"Latent dimension: {latent_dim}")

    generator = Generator(latent_dim, num_classes, X_train_undersampled.shape[1])
    discriminator = Discriminator(X_train_undersampled.shape[1], num_classes)

    print("\nGenerator architecture:")
    print(generator)
    print("\nDiscriminator architecture:")
    print(discriminator)

    history = train_gan(generator, discriminator, train_loader, val_loader,
                        device, model_saver, num_epochs=num_epochs, save_interval=10,
                        patience=40, min_delta=0.0005)

    X_synthetic, y_synthetic = generate_synthetic_data(generator, 30000, num_classes, device)

    return generator, discriminator, history, X_synthetic, y_synthetic

generator, discriminator, history, X_synthetic, y_synthetic = main()

Using device: cpu
Training data shapes - X: (360000, 60), y: (360000,)
Number of features: 60
Number of classes: 10
Latent dimension: 200

Generator architecture:
Generator(
  (initial): Linear(in_features=210, out_features=128, bias=True)
  (hidden1): Linear(in_features=128, out_features=256, bias=True)
  (hidden2): Linear(in_features=256, out_features=512, bias=True)
  (output): Linear(in_features=512, out_features=60, bias=True)
  (leaky): LeakyReLU(negative_slope=0.2)
)

Discriminator architecture:
Discriminator(
  (initial): Linear(in_features=70, out_features=512, bias=True)
  (hidden1): Linear(in_features=512, out_features=256, bias=True)
  (hidden2): Linear(in_features=256, out_features=128, bias=True)
  (output): Linear(in_features=128, out_features=1, bias=True)
  (leaky): LeakyReLU(negative_slope=0.2)
)
Epoch [1/200] G_loss: 1.0700 D_loss: 0.5564
Epoch [2/200] G_loss: 1.3789 D_loss: 0.4911
Epoch [3/200] G_loss: 1.3280 D_loss: 0.4956
Epoch [4/200] G_loss: 1.3330 D_loss: 0.486

TypeError: generate_synthetic_data() missing 2 required positional arguments: 'X_train' and 'y_train'

In [52]:
def generate_synthetic_data(generator, num_samples, label_dim, device, X_train, y_train):
    # Ensure generator is on the proper device.
    generator = generator.to(device)
    generator.eval()
    synthetic_data = []
    synthetic_labels = []

    with torch.no_grad():
        for class_idx in range(label_dim):
            labels = torch.zeros(num_samples, label_dim, device=device)
            labels[:, class_idx] = 1

            z = torch.randn(num_samples, generator.initial.in_features - label_dim, device=device)
            fake_data = generator(z, labels).cpu().numpy()

            synthetic_data.append(fake_data)
            synthetic_labels.append(np.full(num_samples, class_idx))

    X_synthetic = pd.DataFrame(np.vstack(synthetic_data), columns=X_train.columns)
    
    if hasattr(y_train, 'columns'):
        label_columns = y_train.columns
    elif hasattr(y_train, 'name') and y_train.name is not None:
        label_columns = [y_train.name]
    else:
        label_columns = ['target']
    
    y_synthetic = pd.DataFrame(np.hstack(synthetic_labels), columns=label_columns)
    
    return X_synthetic, y_synthetic

# Define hyperparameters and data (assuming these are defined somewhere)
latent_dim = 200
num_classes = 10  
input_dim = X_train_undersampled.shape[1] 

# Create the generator architecture (must match training)
generator = Generator(latent_dim, num_classes, input_dim)

# Load the saved generator weights (update the path as needed)
generator_path = Path('./models/gan_model/quick_save/generator.pt')
generator.load_state_dict(torch.load(generator_path, map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu")))
generator.eval()

# Use the synthetic data generation function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_synthetic, y_synthetic = generate_synthetic_data(generator, 30000, num_classes, device, X_train_undersampled, y_train_undersampled)

print("Synthetic data generated successfully!")

Synthetic data generated successfully!


In [53]:
print(f"Synthetic data shape: {X_synthetic.shape}")
print("Training XGBoost on synthetic data...")
train_xgboost(X_synthetic, y_synthetic, X_val, y_val, X_test, y_test)

Synthetic data shape: (300000, 60)
Training XGBoost on synthetic data...

Evaluating on Validation Set:

Classification Report:
                precision    recall  f1-score   support

      Analysis     0.0246    0.0075    0.0115       401
      Backdoor     0.0000    0.0000    0.0000       349
           DoS     0.2138    0.0126    0.0239      2453
      Exploits     0.6109    0.4358    0.5087      6679
       Fuzzers     0.0624    0.7440    0.1151      3637
       Generic     0.0000    0.0000    0.0000     32322
        Normal     1.0000    0.8696    0.9302    332815
Reconnaissance     0.6258    0.5724    0.5980      2098
     Shellcode     0.0270    0.9956    0.0525       227
         Worms     0.0248    0.4231    0.0469        26

      accuracy                         0.7782    381007
     macro avg     0.2589    0.4061    0.2287    381007
  weighted avg     0.8897    0.7782    0.8261    381007


Confusion Matrix:
[[     3      0      3     88    307      0      0      0      0  

In [56]:
def augment_data(X_train, y_train, X_synthetic, y_synthetic):
    """
    Augments the original training data with generated synthetic data.

    Parameters:
        X_train (pd.DataFrame): Original training features.
        y_train (pd.Series or np.array): Original training labels.
        X_synthetic (np.array or pd.DataFrame): Synthetic features generated by the CGAN.
        y_synthetic (np.array or pd.Series): Synthetic labels corresponding to X_synthetic.

    Returns:
        X_augmented (pd.DataFrame): Augmented training features.
        y_augmented (np.array): Augmented training labels.
    """
    # If X_synthetic is not a DataFrame, convert it using X_train's columns
    if not isinstance(X_synthetic, pd.DataFrame):
        X_synthetic = pd.DataFrame(X_synthetic, columns=X_train.columns)

    # Concatenate feature DataFrames along rows
    X_augmented = pd.concat([X_train, X_synthetic], axis=0)
    
    # Ensure y_train and y_synthetic are 1D arrays before concatenating
    y_train = np.array(y_train).flatten()
    y_synthetic = np.array(y_synthetic).flatten()
    y_augmented = np.hstack((y_train, y_synthetic))

    return X_augmented, y_augmented

# Use the original DataFrame rather than converting to NumPy arrays
X_augmented, y_augmented = augment_data(X_train_smote, y_train_smote, X_synthetic, y_synthetic)

print(f"Original training data shape: {X_train.shape}")
print(f"Augmented training data shape: {X_augmented.shape}")

Original training data shape: (1778032, 60)
Augmented training data shape: (2243971, 60)


In [60]:
import joblib

# Function to train and validate the XGBoost model and save it to a file
def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test, model_filename):
    model = XGBClassifier(tree_method='hist', device='cuda', eval_metric="mlogloss",
                          random_state=42, num_classes=10, objective="multi:softmax", max_depth=15,
                          n_estimators=200, learning_rate=0.2)

    # Training the model without early stopping
    start_time = time.time()
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose = False)
    training_time = time.time() - start_time

    print("\nEvaluating on Validation Set:")
    evaluate_xgb(model, X_val, y_val)

    print("\nEvaluating on Test Set:")
    evaluate_xgb(model, X_test, y_test)

    # Save the trained model to a file
    joblib.dump(model, model_filename)
    print(f"\nModel saved to {model_filename}")

    print('\nTraining Time: {:.2f} seconds'.format(training_time))

# Function to evaluate the model (for both validation and test sets)
def evaluate_xgb(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred, digits=4, target_names=[class_mapping[i] for i in range(len(class_mapping))]))

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_eval, y_pred)
    print(cm)

    print("\nROC-AUC Score:", roc_auc_score(pd.get_dummies(y_eval), model.predict_proba(X_eval), multi_class="ovr"))
    print("\nMatthews Correlation Coefficient:", matthews_corrcoef(y_eval, y_pred))

print("Training XGBoost on augmented data...")
train_xgboost(X_augmented, y_augmented, X_val, y_val, X_test, y_test,'unsw_xgb.pkl')

Training XGBoost on augmented data...

Evaluating on Validation Set:

Classification Report:
                precision    recall  f1-score   support

      Analysis     0.2101    0.1870    0.1979       401
      Backdoor     0.1109    0.1633    0.1321       349
           DoS     0.3386    0.7216    0.4609      2453
      Exploits     0.8343    0.5821    0.6858      6679
       Fuzzers     0.8058    0.6995    0.7489      3637
       Generic     0.9977    0.9878    0.9927     32322
        Normal     0.9977    0.9982    0.9979    332815
Reconnaissance     0.9021    0.7860    0.8400      2098
     Shellcode     0.7875    0.9471    0.8600       227
         Worms     0.5500    0.4231    0.4783        26

      accuracy                         0.9825    381007
     macro avg     0.6535    0.6496    0.6395    381007
  weighted avg     0.9864    0.9825    0.9836    381007


Confusion Matrix:
[[    75     62    216      4      3      0     41      0      0      0]
 [    39     57    222     2

In [61]:
def save_test_data(X_test, y_test, filename="test_data.csv"):
    """
    Saves the X_test features and y_test labels as a single CSV file.

    Parameters:
        X_test (pd.DataFrame): Test features.
        y_test (pd.Series or np.array): Test labels.
        filename (str): The name of the output CSV file.
    """
    # If y_test is a NumPy array, convert it to a DataFrame
    if not isinstance(y_test, pd.Series):
        y_test = pd.Series(y_test, name='Label')

    # Concatenate X_test and y_test along columns
    test_data = pd.concat([X_test, y_test], axis=1)

    # Save the DataFrame to a CSV file
    test_data.to_csv(filename, index=False)
    print(f"Test data saved to {filename}")

save_test_data(X_test, y_test, "unsw_test.csv")

Test data saved to unsw_test.csv


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from tqdm import tqdm
import multiprocessing as mp
from multiprocessing import Pool
import os

# Optimizing the CGANEvaluator class
class CGANEvaluator:
    def __init__(self, generator, original_data, original_labels, device, n_jobs=-1):
        self.generator = generator
        self.original_data = original_data
        self.original_labels = original_labels
        self.device = device
        self.n_jobs = n_jobs  # Number of jobs for parallel execution
        self.scaler = StandardScaler()
        self.original_data_scaled = self.scaler.fit_transform(original_data)

    def generate_samples_for_class(self, class_idx, samples_per_class, unique_labels):
        """Generate synthetic samples for each class"""
        labels = torch.zeros(samples_per_class, len(unique_labels), device=self.device)
        labels[:, class_idx] = 1

        # Generate latent vectors
        z = torch.randn(samples_per_class,
                        self.generator.latent_dim,
                        device=self.device)

        # Generate fake data
        fake_data = self.generator(z, labels).cpu().numpy()

        synthetic_labels = np.full(samples_per_class, class_idx)
        return fake_data, synthetic_labels

    def generate_samples(self, samples_per_class):
        """Generate synthetic samples for each class in parallel"""
        self.generator.eval()
        unique_labels = np.unique(self.original_labels)

        # Adjust number of jobs
        if self.n_jobs == -1:
            self.n_jobs = os.cpu_count()  # Use all available cores
        elif self.n_jobs <= 0:
            self.n_jobs = 1  # Use at least 1 process

        # Generate in batches
        batch_size = 5000  # Process in batches to save memory and reduce overhead
        all_synthetic_data = []
        all_synthetic_labels = []

        for start in range(0, samples_per_class, batch_size):
            end = min(start + batch_size, samples_per_class)
            with Pool(processes=self.n_jobs) as pool:
                results = pool.starmap(self.generate_samples_for_class,
                                       [(class_idx, end - start, unique_labels) for class_idx in unique_labels])

            batch_data = np.vstack([result[0] for result in results])
            batch_labels = np.hstack([result[1] for result in results])

            all_synthetic_data.append(batch_data)
            all_synthetic_labels.append(batch_labels)

        # Combine all batches
        synthetic_data = np.vstack(all_synthetic_data)
        synthetic_labels = np.hstack(all_synthetic_labels)

        # Scale the synthetic data
        self.synthetic_data_scaled = self.scaler.transform(synthetic_data)

        return synthetic_data, synthetic_labels

    def calculate_distribution_metrics_for_feature(self, feature_idx):
        """Calculate KS and Wasserstein for each feature"""
        ks_stat, _ = ks_2samp(self.original_data_scaled[:, feature_idx],
                               self.synthetic_data_scaled[:, feature_idx])
        w_dist = wasserstein_distance(self.original_data_scaled[:, feature_idx],
                                      self.synthetic_data_scaled[:, feature_idx])
        return ks_stat, w_dist

    def calculate_distribution_metrics(self):
        """Calculate distribution similarity metrics in parallel"""
        with Pool(processes=self.n_jobs) as pool:
            results = pool.map(self.calculate_distribution_metrics_for_feature, range(self.original_data.shape[1]))

        ks_stats, wasserstein_stats = zip(*results)
        metrics = {
            'ks_test_mean': np.mean(ks_stats),
            'ks_test_std': np.std(ks_stats),
            'wasserstein_mean': np.mean(wasserstein_stats),
            'wasserstein_std': np.std(wasserstein_stats),
            'original_silhouette': silhouette_score(self.original_data_scaled, self.original_labels),
            'synthetic_silhouette': silhouette_score(self.synthetic_data_scaled, self.synthetic_labels)
        }

        return metrics

    def visualize_distributions(self):
        """Create visualizations comparing original and synthetic data"""
        tsne = TSNE(n_components=2, random_state=42, n_jobs=1)  # Use only 1 CPU for TSNE

        # Combine data for t-SNE
        combined_data = np.vstack([self.original_data_scaled, self.synthetic_data_scaled])
        combined_labels = np.hstack([self.original_labels, self.synthetic_labels])
        data_type = np.array(['Original'] * len(self.original_labels) + ['Synthetic'] * len(self.synthetic_labels))

        tsne_results = tsne.fit_transform(combined_data)

        # Create figure with subplots
        fig = plt.figure(figsize=(20, 10))

        # t-SNE plot by class
        plt.subplot(1, 2, 1)
        scatter = plt.scatter(tsne_results[:, 0], tsne_results[:, 1],
                              c=combined_labels, cmap='tab10', alpha=0.6)
        plt.colorbar(scatter)
        plt.title('t-SNE Visualization by Class')

        # t-SNE plot by data type
        plt.subplot(1, 2, 2)
        for data_type_name in ['Original', 'Synthetic']:
            mask = data_type == data_type_name
            plt.scatter(tsne_results[mask, 0], tsne_results[mask, 1], label=data_type_name, alpha=0.6)
        plt.legend()
        plt.title('t-SNE Visualization by Data Type')

        plt.tight_layout()
        plt.show()

        # Feature distributions
        num_features = min(6, self.original_data.shape[1])  # Show first 6 features
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.ravel()

        for i in range(num_features):
            sns.kdeplot(data=self.original_data[:, i], ax=axes[i],
                        label='Original', alpha=0.6)
            sns.kdeplot(data=self.synthetic_data[:, i], ax=axes[i],
                        label='Synthetic', alpha=0.6)
            axes[i].set_title(f'Feature {i+1} Distribution')
            axes[i].legend()

        plt.tight_layout()
        plt.show()

    def class_balance_analysis(self):
        """Analyze class distribution balance"""
        orig_class_counts = pd.Series(self.original_labels).value_counts()
        syn_class_counts = pd.Series(self.synthetic_labels).value_counts()

        # Plot class distributions
        plt.figure(figsize=(12, 6))

        x = np.arange(len(orig_class_counts))
        width = 0.35

        plt.bar(x - width/2, orig_class_counts, width, label='Original')
        plt.bar(x + width/2, syn_class_counts, width, label='Synthetic')

        plt.xlabel('Class')
        plt.ylabel('Count')
        plt.title('Class Distribution Comparison')
        plt.legend()
        plt.xticks(x)
        plt.show()

        return {
            'original_distribution': orig_class_counts,
            'synthetic_distribution': syn_class_counts
        }

# Main function to evaluate the performance of the CGAN model
def evaluate_cgan(generator, original_data, original_labels, device, samples_per_class=50000, n_jobs=-1):
    """Main function to evaluate CGAN performance"""
    evaluator = CGANEvaluator(generator, original_data, original_labels, device, n_jobs=n_jobs)

    # Generate synthetic samples
    print("Generating synthetic samples...")
    synthetic_data, synthetic_labels = evaluator.generate_samples(samples_per_class)

    # Calculate metrics
    print("\nCalculating distribution metrics...")
    metrics = evaluator.calculate_distribution_metrics()

    # Print metrics
    print("\nEvaluation Metrics:")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")

    # Generate visualizations
    print("\nGenerating visualizations...")
    evaluator.visualize_distributions()

    # Analyze class balance
    print("\nAnalyzing class balance...")
    balance_metrics = evaluator.class_balance_analysis()

    return synthetic_data, synthetic_labels, metrics, evaluator

# Set multiprocessing start method to 'spawn' (this is crucial for CUDA)
if __name__ == '__main__':
    mp.set_start_method('spawn', force=True)

    # Ensure that at least one process is used and we don't use zero processes
    num_processes = os.cpu_count() or 1  # This ensures at least one process is used

    # Your code where you run evaluation
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Proceed with your CGAN evaluation
    X_synthetic, y_synthetic, metrics, evaluator = evaluate_cgan(
        generator=generator,
        original_data=X_train_smote,  # Your original training data
        original_labels=y_train_smote,  # Your original training labels
        device=device,
        samples_per_class=20000
    )